# A0 Action Distribution Plots

A0-only action-distribution notebook backed by `action_distribution_metrics.py`. Tables are kept available in variables but only figures are displayed.


In [25]:
from pathlib import Path
import importlib
import sys

for candidate in [Path.cwd(), *Path.cwd().parents]:
    helpers = candidate / "helpers"
    repo_helpers = candidate / "Topology_Task" / "analysis" / "metrics" / "helpers"
    if helpers.exists() and (helpers / "action_distribution_metrics.py").exists():
        sys.path.insert(0, str(helpers))
        break
    if repo_helpers.exists() and (repo_helpers / "action_distribution_metrics.py").exists():
        sys.path.insert(0, str(repo_helpers))
        break
else:
    raise FileNotFoundError("Could not locate Topology_Task/analysis/metrics/helpers")

import action_distribution_metrics as adm
adm = importlib.reload(adm)
import wandb_metrics as wm
wm = importlib.reload(wm)
print("action_distribution_metrics:", adm.__file__)
print("wandb_metrics:", wm.__file__)


action_distribution_metrics: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/analysis/metrics/helpers/action_distribution_metrics.py
wandb_metrics: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/analysis/metrics/helpers/wandb_metrics.py


## Choose A0 Runs To Load Or Download


In [26]:
import importlib
wm = importlib.reload(wm)
adm = importlib.reload(adm)

# Pick A0 config folder(s) under Topology_Task/configs, or use None / "None" / "all" for every run.
# Examples:
# CONFIG_FOLDERS_TO_DOWNLOAD = "a0_hvg"
# CONFIG_FOLDERS_TO_DOWNLOAD = ["a0_hvg", "a0_sparse16"]
# CONFIG_FOLDERS_TO_DOWNLOAD = "a0_hvg,a0_sparse16,a0_aib"
# CONFIG_FOLDERS_TO_DOWNLOAD = "all"
CONFIG_FOLDERS_TO_DOWNLOAD = ["a0_hvg", "a0_sparse16", "a0_aib", "a0_test_rerun"]

# False only reads local cached histories. True downloads missing matching histories from W&B.
DOWNLOAD_MISSING_FROM_WANDB = False

# Useful when W&B has newer data than the local cache, especially for old scan_history fallback caches.
# This only has an effect when DOWNLOAD_MISSING_FROM_WANDB is True.
REFRESH_SCAN_HISTORY_FALLBACKS = True

# Heavier option: replace every selected local cache from W&B.
FORCE_REFRESH_CACHE = False

wm.configure_run_filter_from_config_folder(CONFIG_FOLDERS_TO_DOWNLOAD)
wm.EXCLUDE_RUN_NAME_REGEX = None
wm.RUN_STATES = None
wm.MAX_RUNS = None
wm.USE_LOCAL_CACHE_ONLY = not DOWNLOAD_MISSING_FROM_WANDB
wm.REFRESH_SCAN_HISTORY_FALLBACKS = bool(DOWNLOAD_MISSING_FROM_WANDB and REFRESH_SCAN_HISTORY_FALLBACKS)
wm.FORCE_REFRESH = bool(DOWNLOAD_MISSING_FROM_WANDB and FORCE_REFRESH_CACHE)
wm.ALLOW_SCAN_HISTORY_FALLBACK = True
wm.refresh_run_filters()

print("RUN_NAME_REGEX =", wm.RUN_NAME_REGEX)
print("USE_LOCAL_CACHE_ONLY =", wm.USE_LOCAL_CACHE_ONLY)
print("REFRESH_SCAN_HISTORY_FALLBACKS =", wm.REFRESH_SCAN_HISTORY_FALLBACKS)
print("FORCE_REFRESH =", wm.FORCE_REFRESH)


Config folder filter:
  - /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/configs/a0_hvg
  - /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/configs/a0_sparse16
  - /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/configs/a0_aib
  - /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/configs/a0_test_rerun
Matched config run-name candidates: 70
RUN_NAME_REGEX = ^\s*(?:a0_hvg_00_baseline_s0|a0_hvg_00_baseline_s1|a0_hvg_00_baseline_s2|a0_hvg_01_eval_rho090_s0|a0_hvg_01_eval_rho090_s1|a0_hvg_01_eval_rho090_s2|a0_hvg_02_gate_final_map_s0|a0_hvg_02_gate_final_map_s1|a0_hvg_02_gate_final_map_s2|a0_hvg_03_gate_hierarchical_s0|a0_hvg_03_gate_hierarchical_s1|a0_hvg_03_gate_hierarchical_s2|a0_hvg_04_eval_local_rho090_s0|a0_hvg_04_eval_local_rho090_s1|a0_hvg_04_eval_local_rho090_s2|a0_sparse16_flat_p000_s0|a0_sparse16_flat_p000_s1|a0_sparse16_flat_p000_s2|a0_sparse16_flat_p001_s0|a0_sparse16_flat_p001_s1|a0_sparse16_flat_p001_s2|a0_sparse16_flat_p003_s0|a0_sp

## Load Or Download Selected W&B Histories


In [27]:
download_data = wm.load_wandb_data()
downloaded_runs_df = download_data.runs_df
print(f"Matching W&B runs: {len(downloaded_runs_df)}")

Project: corentin-plumet-epfl/Grid2Op
Task dir: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task
Cache mode: full
Cache dir: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_cache
Local-only mode: True
Force refresh: False
Refresh scan-history fallbacks: False
Selected 59 cached runs from /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_cache/full_history
state
finished    40
running     16
crashed      3
History artifact setup: local_only=True, runs_df=59
[ 1/59] loading artifact cache: a0_aib_00_flat_local_t020_s0
    loaded 374 rows, 140 columns from /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_cache/full_history/a0_aib_00_flat_local_t020_s0__MAPPO_bus14_T_0_0__I__1782409828_44829/history.parquet in 0.1s
[ 2/59] loading artifact cache: a0_aib_00_flat_local_t020_s1
    loaded 390 rows, 140 columns from /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_cache/full_history/a0_aib

## Load Cached Histories And Shared Tables


In [28]:
# Use final evaluation action metrics for the action-distribution plots.
# Set ACTION_METRIC_SOURCE = "train" to reproduce rollout-training plots.
ACTION_METRIC_SOURCE = "eval"
EVAL_METRIC_SPLIT = "test"

# Final-window plots average the last logged points at or before this step.
# Use None to treat each run's actual last logged point as final.
FINAL_SUMMARY_STEP_M = None

ctx = adm.load_action_distribution_context(
    experiment_folders=CONFIG_FOLDERS_TO_DOWNLOAD,
    action_metric_source=ACTION_METRIC_SOURCE,
    eval_split=EVAL_METRIC_SPLIT,
    final_summary_step_m=FINAL_SUMMARY_STEP_M,
    save_figures=True,
    show_figures=False,
)

Expected configs: 70
Cached expected runs: 59 / 70
Missing cached histories for these expected configs:
[  1/59] loading a0_aib_00_flat_local_t020_s0
[  2/59] loading a0_aib_00_flat_local_t020_s1
[  3/59] loading a0_aib_00_flat_local_t020_s2
[  4/59] loading a0_aib_01_flat_local_t010_s0
[  5/59] loading a0_aib_01_flat_local_t010_s1
[  6/59] loading a0_aib_01_flat_local_t010_s2
[  7/59] loading a0_aib_02_flat_local_t035_s0
[  8/59] loading a0_aib_02_flat_local_t035_s1
[  9/59] loading a0_aib_02_flat_local_t035_s2
[ 10/59] loading a0_aib_03_gate_hgreedy_sep_local_t020_s0
[ 11/59] loading a0_aib_03_gate_hgreedy_sep_local_t020_s1
[ 12/59] loading a0_aib_03_gate_hgreedy_sep_local_t020_s2
[ 13/59] loading a0_aib_04_flat_nonidle_t020_s0
[ 14/59] loading a0_aib_04_flat_nonidle_t020_s1
[ 15/59] loading a0_aib_04_flat_nonidle_t020_s2
[ 16/59] loading a0_hvg_00_baseline_s0
[ 17/59] loading a0_hvg_00_baseline_s1
[ 18/59] loading a0_hvg_00_baseline_s2
[ 19/59] loading a0_hvg_01_eval_rho090_s0
[ 20/

## Last-5logged Action 0 Fraction By Run And Agent


In [29]:
last5_action0 = adm.last5_logged_action0_fraction_by_agent_all_runs(ctx)
adm.display_metric_figures(last5_action0)


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/final_action0_fraction_by_agent_all_runs_a0_hvg.html
Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/final_action0_fraction_by_agent_all_runs_a0_sparse16.html
Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/final_action0_fraction_by_agent_all_runs_a0_aib.html
Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/final_action0_fraction_by_agent_all_runs_a0_test_rerun.html


## Seed-Aggregated Last-5-Logged Action-0 Fraction By Run And Agent


In [30]:
seed_action0 = adm.seed_aggregated_last5_logged_action0_fraction_by_agent(ctx)
adm.display_metric_figures(seed_action0)


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/seed_aggregated_final_action0_fraction_by_agent_a0_hvg.html
Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/seed_aggregated_final_action0_fraction_by_agent_a0_sparse16.html
Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/seed_aggregated_final_action0_fraction_by_agent_a0_aib.html
Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/seed_aggregated_final_action0_fraction_by_agent_a0_test_rerun.html


## Survival vs Action-0 Over Time


In [31]:
survival_action = adm.survival_vs_action0_non_idle_over_time(ctx)
adm.display_metric_figures(survival_action)


Missing survival or action-behavior data for intervention_gate_15
Missing survival or action-behavior data for phase4_sparse_control_16
Missing survival or action-behavior data for heuristic_vs_gate_s0_s1_s2
Missing survival or action-behavior data for adaptive_intervention_budget_7
Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/survival_vs_action_behavior_a0_hvg.html
Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/survival_vs_action_behavior_a0_sparse16.html
Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/survival_vs_action_behavior_a0_aib.html
Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/survival_vs_action_behavior_a0_test_rerun.html


## Action-0 / Survival Tradeoff


In [32]:
action0_survival = adm.action0_survival_tradeoff(ctx)
adm.display_metric_figures(action0_survival)


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/final_action0_vs_survival_tradeoff_a0_aib.html
Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/final_action0_vs_survival_tradeoff_a0_hvg.html
Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/final_action0_vs_survival_tradeoff_a0_sparse16.html
Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/final_action0_vs_survival_tradeoff_a0_test_rerun.html


## Entropy Collapse vs Action-0 Confidence


In [33]:
entropy_action0 = adm.entropy_collapse_vs_action0_confidence(ctx)
adm.display_metric_figures(entropy_action0)


No evaluation entropy metrics found; skipping entropy/action-0 plots.


## Agent Non-Idle Imbalance


In [34]:
agent_imbalance = adm.agent_non_idle_imbalance(ctx)
adm.display_metric_figures(agent_imbalance)


No agent imbalance time-series data for intervention_gate_15
No agent imbalance time-series data for phase4_sparse_control_16
No agent imbalance time-series data for heuristic_vs_gate_s0_s1_s2
No final agent imbalance data for intervention_gate_15
No final agent imbalance data for phase4_sparse_control_16
No final agent imbalance data for heuristic_vs_gate_s0_s1_s2
No agent imbalance time-series data for adaptive_intervention_budget_7
No final agent imbalance data for adaptive_intervention_budget_7
Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/agent_non_idle_imbalance_timeseries_a0_hvg.html
Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/agent_non_idle_imbalance_timeseries_a0_sparse16.html
Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/agent_non_idle_imbalance_timeseries_a0_aib.html
Saved: /Use

## Joint Action Coordination


In [35]:
joint_coordination = adm.joint_action_coordination(ctx)
adm.display_metric_figures(joint_coordination)


No scalar evaluation joint-action metrics found; skipping joint coordination plots. Eval joint coordination needs eval non_idle_agents_count_* scalars or trace-table processing.


## Gate Probability vs Actual Intervention Fraction


In [36]:
gate_calibration = adm.gate_probability_vs_actual_intervention_fraction(ctx)
adm.display_metric_figures(gate_calibration)


No evaluation gate-probability metrics found; skipping gate probability vs actual plots. Eval logs final gate intervention fractions, but not gate probabilities.
